# 10 — Datetime & Categorical Features: The Last Mile of Time Series FE

**Companion notebook to Article 10 of the *Time Series Feature Engineering* series.**

📖 Read the article: [Medium](https://medium.com/@asidd24) · [LinkedIn Newsletter](https://www.linkedin.com/newsletters/time-series-engineered-7485187061080137728/)

---

This notebook walks through every code example and figure from Article 10, demonstrating:

1. **Event features** — a holiday flag, countdown / count-up, and payday / business-day structure
2. **Categorical features in a panel** — six ways to hand `store_id` to a tree model
3. **Target encoding leakage** — why it's a fitted feature, and the diagnostic for catching it
4. **One feature matrix, with a leakage ledger** — assembling nine articles' worth of feature families
5. **The capstone ablation** — adding every block one at a time on a 200-store synthetic panel

The article's own published numbers came from a hand-tuned synthetic generator whose exact
parameters aren't part of this repo. This notebook reproduces the *mechanism* behind every
claim on its own panel — expect the same conclusions, not bit-identical numbers. The
companion script [`article10_medium/make_figures.py`](https://github.com/Ameen2488/time-series-feature-engineering)
runs the same experiments end to end and saves the article's figures.

## 1. Setup & Environment

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from src.data import generate_store_panel
from src.features import (
    add_business_day_features,
    add_event_features,
    add_holiday_distance_features,
    add_lag_features_horizon_aware,
    add_rolling_features,
    add_seasonal_features,
    as_categorical,
    target_encode_expanding,
    target_encode_naive,
    target_encode_out_of_fold,
)
from src.metrics import wmape

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (12, 4),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})
np.random.seed(42)

BLUE, GREEN, RED, GRAY, ORANGE, PURPLE, DARK = (
    '#1f77b4', '#2ca02c', '#d62728', '#9aa0a6', '#ff7f0e', '#9467bd', '#333333'
)
H = 7            # forecast horizon — 7-day-ahead direct, matching the article
N_STORES = 200   # the article's panel size; drop this for a faster interactive run

## 2. Data: A 200-Store Panel With Events Baked In

`generate_store_panel` builds a daily sales panel across many stores, 2022–2024, with a
demand ramp-up / spike / dip around a synthetic **"Black Friday"** (the 4th Friday of
November, computed from a rule — not a hardcoded date, so it lands on a different
day-of-year every year) plus smaller bumps around a few fixed-date holidays and a mild
payday effect near the business-day month-end.

`generate_holiday_calendar` returns the dates it used — this is the "calendar known in
advance" that every event feature below is built from.

In [ ]:
panel, holidays = generate_store_panel(n_stores=N_STORES, seed=7)
print(f"{panel['store_id'].nunique()} stores x {panel['date'].nunique()} days = {len(panel):,} rows")
print(f"{panel['date'].min().date()} -> {panel['date'].max().date()}")
holidays

## 3. Figure 1 — Event Effects: Ramp-Up, Spike, Dip

Average each store's sales against its own mean, then look at that ratio in the days
immediately around Black Friday. A fixed-date seasonality feature could never produce this
shape on its own, because the date itself moves every year.

In [ ]:
bf_dates = holidays.loc[holidays['name'] == 'black_friday', 'date']
store_mean = panel.groupby('store_id')['sales'].transform('mean')
rel = (panel['sales'] / store_mean).to_numpy()
dates = panel['date'].to_numpy()

window = range(-14, 8)
curve = []
for off in window:
    shifted = bf_dates.to_numpy() + np.timedelta64(off, 'D')
    mask = np.isin(dates, shifted)
    curve.append(rel[mask].mean())

fig, ax = plt.subplots(figsize=(12, 5))
colors = [RED if x == 0 else (ORANGE if x < 0 else BLUE) for x in window]
ax.bar(list(window), curve, color=colors, alpha=0.85)
ax.axhline(1.0, color=DARK, lw=1.2, ls='--', label="Store's average day")
ax.axvline(0, color=RED, lw=1, alpha=0.4)
ax.set_xlabel('Days relative to Black Friday')
ax.set_ylabel('Demand multiplier (sales / store average)')
ax.set_title('Demand Ramps Up Before an Event, Spikes, Then Dips', loc='left')
ax.legend(loc='upper left')
plt.show()

print(f"Peak multiplier on the day: {curve[list(window).index(0)]:.2f}x")

## 4. The Event Feature Toolkit

Three building blocks, matching the article's Part 2:

- **`add_holiday_distance_features`** — a flag plus signed distance to the nearest known
  event, clipped at `max_distance` days. The distance columns are what let a tree learn the
  ramp-up and the hangover; a flag alone can only say "today is special."
- **`add_business_day_features`** — month-end / business-day structure, for payroll-driven
  demand.
- **`add_event_features`** — both of the above in one call. Build this on the full frame
  before splitting: every column is a function of the date and a calendar known in advance,
  so there is nothing to leak.

In [ ]:
sample = panel[panel['store_id'] == 'S0000'].copy()
sample = add_event_features(sample, holidays, date_col='date', max_distance=60)
sample[['date', 'sales', 'is_holiday', 'days_to_holiday', 'days_since_holiday',
        'is_month_end_bday', 'bdays_to_month_end']].iloc[318:326]

## 5. Figure 2 — Event Feature Gains

In [ ]:
def build_calendar_block(panel):
    cal = pd.DataFrame({'date': pd.date_range(panel['date'].min(), panel['date'].max(), freq='D')})
    cal['t_index'] = np.arange(len(cal))
    cal = add_seasonal_features(cal.set_index('date'), yearly_k=6).reset_index()
    cal = add_event_features(cal, holidays, date_col='date', max_distance=60)
    cal['is_holiday_window'] = (cal['days_to_holiday'] <= 3) | (cal['days_since_holiday'] <= 3)
    return cal

def build_store_block(panel):
    frames = []
    for _, g in panel.groupby('store_id', sort=False):
        g = g.sort_values('date').reset_index(drop=True)
        g = add_lag_features_horizon_aware(g, 'sales', lags=[7, 14, 21, 28], forecast_horizon=H, prefix='lag')
        g = add_rolling_features(g, 'sales', windows=[7, 14, 28], functions=['mean', 'std'], min_horizon=H, prefix='roll')
        frames.append(g)
    return pd.concat(frames, ignore_index=True)

cal = build_calendar_block(panel)
feat = build_store_block(panel).merge(cal, on='date', how='left')

LAG_COLS = ['lag_7', 'lag_14', 'lag_21', 'lag_28']
ROLL_COLS = [c for c in feat.columns if c.startswith('roll_')]
SEASON_COLS = [c for c in cal.columns if c.startswith('dow_') or c.startswith('fy_')]
HOLIDAY_COLS = ['is_holiday', 'days_to_holiday', 'days_since_holiday']
BDAY_COLS = ['is_month_end_bday', 'bdays_to_month_end']

print(f"{len(LAG_COLS)} lag, {len(ROLL_COLS)} rolling, {len(SEASON_COLS)} seasonal, "
      f"{len(HOLIDAY_COLS)} holiday, {len(BDAY_COLS)} business-day columns")

In [ ]:
def fit_wmape(train, test, cols, cat_cols=None):
    cat_cols = cat_cols or []
    tr = train.dropna(subset=cols)
    model = lgb.LGBMRegressor(
        objective='regression', n_estimators=220, num_leaves=31,
        learning_rate=0.08, min_child_samples=30, subsample=0.9,
        colsample_bytree=0.9, random_state=7, verbosity=-1,
    )
    model.fit(tr[cols], tr['sales'], categorical_feature=cat_cols or 'auto')
    return np.clip(model.predict(test[cols]), 0, None)

def slice_wmape(test, pred):
    all_days = wmape(test['sales'].values, pred) * 100
    hw = test['is_holiday_window'].values
    holiday_days = wmape(test['sales'].values[hw], pred[hw]) * 100
    return all_days, holiday_days

train = feat[feat['date'].dt.year.isin([2022, 2023])]
test = feat[feat['date'].dt.year == 2024]

base_cols = LAG_COLS + ROLL_COLS + ['t_index'] + SEASON_COLS
event_steps = [
    ('No event features', base_cols),
    ('+ is_holiday flag', base_cols + ['is_holiday']),
    ('+ countdown / count-up', base_cols + HOLIDAY_COLS),
    ('+ payday / business-day', base_cols + HOLIDAY_COLS + BDAY_COLS),
]

event_results = {}
for name, cols in event_steps:
    pred = fit_wmape(train, test, cols)
    event_results[name] = slice_wmape(test, pred)

pd.DataFrame(event_results, index=['WMAPE all days (%)', 'WMAPE holiday windows (%)']).T.round(2)

In [ ]:
labels = [s[0] for s in event_steps]
all_vals = [event_results[l][0] for l in labels]
hol_vals = [event_results[l][1] for l in labels]

fig, ax = plt.subplots(figsize=(11, 5.4))
x = np.arange(len(labels))
w = 0.35
ax.bar(x - w/2, all_vals, width=w, color=BLUE, alpha=0.88, label='All test days')
ax.bar(x + w/2, hol_vals, width=w, color=RED, alpha=0.88, label='Holiday windows (±3 days)')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=10, ha='right')
ax.set_ylabel('WMAPE (%)')
ax.set_title('Error Concentrates on Events, and So Does the Fix', loc='left')
ax.legend(loc='upper right')
plt.show()

## 6. Categorical Features in a Panel

A panel brings a column single series never had: `store_id`. Six ways to hand it to
LightGBM:

- **None** — don't encode it at all (the control)
- **Integer** — arbitrary integer codes
- **One-hot** — a dummy column per store
- **LightGBM native** — pandas `category` dtype, no manual encoding
- **Target (naive)** — `groupby('store_id')['sales'].mean()`, fit on train and applied as-is
- **Target (expanding)** — `target_encode_expanding`, the leak-safe version

Compared under two settings: **with lags present** (the normal case) and **without lags**,
restricted to stores that never appear in training at all — a genuine cold start, not just
an ablation.

In [ ]:
rng = np.random.default_rng(7)
all_stores = sorted(feat['store_id'].unique())
holdout_stores = list(rng.choice(all_stores, size=20, replace=False))

is_2022_23 = feat['date'].dt.year.isin([2022, 2023])
is_2024 = feat['date'].dt.year == 2024
is_holdout = feat['store_id'].isin(holdout_stores)

event_cols = HOLIDAY_COLS + BDAY_COLS
with_lags_cols = LAG_COLS + ROLL_COLS + ['t_index'] + SEASON_COLS + event_cols
without_lags_cols = ['t_index'] + SEASON_COLS + event_cols

settings = {
    'with_lags': (feat[is_2022_23], feat[is_2024], with_lags_cols),
    'without_lags': (feat[is_2022_23 & ~is_holdout], feat[is_2024 & is_holdout], without_lags_cols),
}

In [ ]:
def encode_variants(train, test):
    variants = {}
    tr, te = train.copy(), test.copy()
    variants['none'] = (tr, te, [], [])

    tr_i = as_categorical(tr, ['store_id'])
    te_i = te.copy()
    te_i['store_id'] = te_i['store_id'].astype(tr_i['store_id'].dtype)
    tr_i['store_int'] = tr_i['store_id'].cat.codes
    te_i['store_int'] = te_i['store_id'].cat.codes
    variants['integer'] = (tr_i, te_i, ['store_int'], [])

    oh_tr = pd.get_dummies(tr['store_id'], prefix='st').astype(int)
    oh_te = pd.get_dummies(te['store_id'], prefix='st').reindex(columns=oh_tr.columns, fill_value=0).astype(int)
    variants['onehot'] = (
        pd.concat([tr.reset_index(drop=True), oh_tr.reset_index(drop=True)], axis=1),
        pd.concat([te.reset_index(drop=True), oh_te.reset_index(drop=True)], axis=1),
        list(oh_tr.columns), [],
    )

    # Fit the category dtype on train, cast test to that SAME dtype — otherwise the
    # integer codes LightGBM's categorical splits rely on silently disagree between
    # train and test. A store never seen in train becomes NaN (missing) under train's
    # category set, which is the correct behaviour for a true cold start.
    tr_n = as_categorical(tr, ['store_id'])
    te_n = te.copy()
    te_n['store_id'] = te_n['store_id'].astype(tr_n['store_id'].dtype)
    variants['native'] = (tr_n, te_n, ['store_id'], ['store_id'])

    global_prior = float(tr['sales'].mean())
    tr_te1 = target_encode_naive(tr, 'store_id', 'sales', new_col='store_te')
    te_te1 = te.copy()
    te_te1['store_te'] = te_te1['store_id'].map(tr.groupby('store_id')['sales'].mean()).fillna(global_prior)
    variants['target_naive'] = (tr_te1, te_te1, ['store_te'], [])

    marked = pd.concat([train.assign(_split='train'), test.assign(_split='test')],
                        ignore_index=True).sort_values(['store_id', 'date'])
    marked = target_encode_expanding(marked, 'store_id', 'sales', new_col='store_te_exp', smoothing=10.0)
    variants['target_expanding'] = (
        marked[marked['_split'] == 'train'], marked[marked['_split'] == 'test'],
        ['store_te_exp'], [],
    )
    return variants

cat_results = {}
for label, (train_raw, test_raw, base_cols) in settings.items():
    train_s = train_raw[base_cols + ['store_id', 'sales', 'date']]
    test_s = test_raw[base_cols + ['store_id', 'sales', 'date']]
    variants = encode_variants(train_s, test_s)
    row = {}
    for vname, (tr, te, extra_cols, cat_cols) in variants.items():
        pred = fit_wmape(tr, te, base_cols + extra_cols, cat_cols=cat_cols)
        row[vname] = wmape(te['sales'].values, pred) * 100
    cat_results[label] = row

pd.DataFrame(cat_results).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order = ['none', 'integer', 'onehot', 'native', 'target_naive', 'target_expanding']
nice = {'none': 'None', 'integer': 'Integer', 'onehot': 'One-hot', 'native': 'LightGBM\nnative',
        'target_naive': 'Target\n(naive)', 'target_expanding': 'Target\n(expanding)'}

for ax, label, title in zip(axes, ['with_lags', 'without_lags'],
                             ['With lags present', 'Without lags (cold start)']):
    vals = [cat_results[label][k] for k in order]
    colors = [GREEN if v == min(vals) else BLUE for v in vals]
    ax.bar([nice[k] for k in order], vals, color=colors, alpha=0.88)
    ax.set_ylabel('WMAPE (%)')
    ax.set_title(title, loc='left', fontsize=13)
fig.suptitle('Categorical Encoding Matters Where History Cannot Speak', y=1.04, fontweight='bold')
plt.show()

print("With lags present, every encoding lands in a narrow band — store identity is already")
print("carried by the lags. Without lags, expanding target encoding wins by a distance.")

## 7. Target Encoding Is a Leakage Machine

Target encoding replaces a category with a statistic of the target for that category —
which makes it a **fitted feature**, subject to the same discipline as any other fitted
feature. The diagnostic below isolates the mechanism: vary the number of training rows per
category level and compare three honestly different quantities.

```python
# 1. NAIVE — the category mean includes the row's own target. Leaks for train evaluation.
df['te'] = df.groupby('store')['y'].transform('mean')

# 2. EXPANDING — only strictly earlier rows. Leak-safe in time. See target_encode_expanding.
df['te'] = df.groupby('store')['y'].transform(lambda s: s.shift(1).expanding().mean())

# 3. FIT-ON-ALL — computed before the train/validation split. Validation leaks.
```

In [ ]:
def make_leakage_data(n_levels, rows_per_level, heldout_per_level=60,
                       level_std=22.0, noise_std=8.0, seed=1):
    rng = np.random.default_rng(seed)
    level_means = rng.normal(100.0, level_std, n_levels)
    levels = np.repeat(np.arange(n_levels), rows_per_level + heldout_per_level)
    y = level_means[levels] + rng.normal(0, noise_std, len(levels))
    df = pd.DataFrame({'level': levels, 'y': y})
    df['is_train'] = (df.groupby('level').cumcount() < rows_per_level)
    return df

grid = [4, 8, 16, 32, 64, 128]
noise_std = 8.0
noise_floor = noise_std * np.sqrt(2 / np.pi)  # E|N(0, noise_std)| — the best any model can do

train_mae, heldout_mae_safe, heldout_mae_fitonall = [], [], []
for n in grid:
    # NOTE: named train_leak/heldout_leak, not train/test — this loop is a small,
    # self-contained diagnostic and must not clobber the panel's train/test frames
    # used later in the capstone ablation (section 9).
    df_leak = make_leakage_data(n_levels=60, rows_per_level=n)
    train_leak, heldout_leak = df_leak[df_leak['is_train']], df_leak[~df_leak['is_train']]

    safe_map = train_leak.groupby('level')['y'].mean()
    train_mae.append(float((train_leak['y'] - train_leak['level'].map(safe_map)).abs().mean()))
    heldout_mae_safe.append(float((heldout_leak['y'] - heldout_leak['level'].map(safe_map)).abs().mean()))

    fitonall_map = df_leak.groupby('level')['y'].mean()  # computed on train + held-out TOGETHER
    heldout_mae_fitonall.append(float((heldout_leak['y'] - heldout_leak['level'].map(fitonall_map)).abs().mean()))

pd.DataFrame({
    'rows_per_level': grid,
    'train_mae_naive': train_mae,
    'heldout_mae_safe': heldout_mae_safe,
    'heldout_mae_fit_on_all': heldout_mae_fitonall,
    'below_noise_floor': [v < noise_floor for v in heldout_mae_fitonall],
}).assign(noise_floor=round(noise_floor, 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

ax = axes[0]
ax.plot(grid, train_mae, marker='o', color=RED, lw=2.2, label='Train MAE (naive, self-evaluated)')
ax.plot(grid, heldout_mae_safe, marker='o', color=GREEN, lw=2.2, label='Held-out MAE (fit on train only)')
ax.axhline(noise_floor, color=DARK, lw=1.4, ls='--', label=f'Noise floor ({noise_floor:.2f})')
ax.set_xscale('log', base=2); ax.set_xticks(grid); ax.set_xticklabels(grid)
ax.set_xlabel('Train rows per category level'); ax.set_ylabel('MAE')
ax.set_title('Naive Encoding Flatters the Training Fit', loc='left', fontsize=13)
ax.legend(loc='upper right')

ax = axes[1]
ax.plot(grid, heldout_mae_fitonall, marker='o', color=RED, lw=2.2,
         label='Held-out MAE (encoding fit on train + held-out)')
ax.axhline(noise_floor, color=DARK, lw=1.4, ls='--', label=f'Noise floor ({noise_floor:.2f})')
ax.fill_between(grid, 0, noise_floor, color=RED, alpha=0.08)
ax.text(grid[0], noise_floor * 0.5, 'Below the noise floor\n= you leaked', color=RED, fontweight='bold')
ax.set_xscale('log', base=2); ax.set_xticks(grid); ax.set_xticklabels(grid)
ax.set_xlabel('Train rows per category level'); ax.set_ylabel('MAE')
ax.set_title('Fitting Before the Split Breaks the Floor', loc='left', fontsize=13)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

**If your validation error is below the noise you can estimate, you have leaked — not
discovered something.** That is the cleanest diagnostic in the article, and it costs nothing
to check: compute the irreducible error from whatever you know about the noise (or a
reasonable lower bound on it), and treat any validation score beneath it as a bug report.

## 8. One Feature Matrix, With a Leakage Ledger

Nine articles produce nine feature families. The test for any column is one question: *at
forecast time, for a date 30 days out, can I fill this value?*

| Block | Source | Rule |
|---|---|---|
| Lags (4 cols) | Article 6 | shift ≥ forecast horizon |
| Window stats (6 cols) | Article 7 | shift ≥ horizon, then roll |
| Trend (`t_index`) | Article 8 | deterministic; build on full frame |
| Seasonality (dummies + Fourier) | Article 9 | build on the full contiguous frame before splitting |
| Holiday / event | This article | calendar **known in advance** |
| Payday / business-day | This article | calendar **known in advance** |
| Store (categorical) | This article | native, or expanding / out-of-fold target encoding |

## 9. The Capstone Ablation

In [ ]:
tr_store = as_categorical(train, ['store_id'])
te_store = test.copy()
te_store['store_id'] = te_store['store_id'].astype(tr_store['store_id'].dtype)

ablation_steps = [
    ('Seasonal naive (lag_7)', None, []),
    ('+ Lags 7/14/21/28', LAG_COLS, []),
    ('+ Window stats', LAG_COLS + ROLL_COLS, []),
    ('+ Trend (t_index)', LAG_COLS + ROLL_COLS + ['t_index'], []),
    ('+ Seasonality', LAG_COLS + ROLL_COLS + ['t_index'] + SEASON_COLS, []),
    ('+ Holidays & countdowns', LAG_COLS + ROLL_COLS + ['t_index'] + SEASON_COLS + HOLIDAY_COLS, []),
    ('+ Payday / business-day', LAG_COLS + ROLL_COLS + ['t_index'] + SEASON_COLS + HOLIDAY_COLS + BDAY_COLS, []),
    ('+ Store (native categorical)',
     LAG_COLS + ROLL_COLS + ['t_index'] + SEASON_COLS + HOLIDAY_COLS + BDAY_COLS + ['store_id'], ['store_id']),
]

ablation_results = {}
for name, cols, cat_cols in ablation_steps:
    if cols is None:
        pred = test['lag_7'].fillna(test['sales'].mean()).to_numpy()
        ablation_results[name] = slice_wmape(test, pred)
    else:
        tr, te = (tr_store, te_store) if cat_cols else (train, test)
        pred = fit_wmape(tr, te, cols, cat_cols=cat_cols)
        ablation_results[name] = slice_wmape(te, pred)

pd.DataFrame(ablation_results, index=['WMAPE all days (%)', 'WMAPE holiday windows (%)']).T.round(2)

In [ ]:
labels = [s[0] for s in ablation_steps]
all_vals = [ablation_results[l][0] for l in labels]
hol_vals = [ablation_results[l][1] for l in labels]

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(labels))
ax.bar(x, all_vals, color=BLUE, alpha=0.85, label='All test days (bars)')
ax2 = ax.twinx()
ax2.plot(x, hol_vals, color=RED, marker='D', ms=9, lw=2, label='Holiday windows (diamonds)')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=22, ha='right', fontsize=9.5)
ax.set_ylabel('WMAPE (%) — all test days')
ax2.set_ylabel('WMAPE (%) — holiday windows', color=RED)
ax.set_title('The Capstone Ablation: Lags and Holidays Are the Big Wins', loc='left')
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labs1 + labs2, loc='upper center', ncol=2)
plt.show()

**A feature block that helps in isolation can hurt in combination.** Watch the trend and
seasonality rows against the step before them — on this panel they are not guaranteed to
move the error down, even though both helped on their own article's series. Add blocks one
at a time, in a fixed order, and keep the table; don't trust a feature because it worked
somewhere else.

## 10. Decision Framework

| Situation | What to do |
|---|---|
| Fixed-date holidays | `is_holiday` flag + countdown / count-up |
| Moving holidays (Easter, lunar) | Event flag + distance; do **not** rely on Fourier |
| Demand ramps before an event | Days-to-event, ±N-day window flags |
| Payroll-driven demand | Last-business-day, mid-month, business days to month end |
| Panel with lags present | Categorical is optional — measure it |
| Cold start / new entities | Categorical encoding is essential; expanding TE or native |
| High-cardinality categories | Smoothed, expanding / OOF target encoding (`target_encode_out_of_fold`) |
| Any fitted feature | Refit inside each fold |
| Validation error below the noise floor | Stop — you leaked |

## 11. Leakage Rules for This Article

**Rule 1 — Events must be known in advance.** A promotion flag back-filled after the fact is
a future leak in a calendar costume. Test: can you fill this column 30 days ahead?

**Rule 2 — Target encoding is a fitted feature.** Expanding or out-of-fold only, refit per
fold — see `target_encode_out_of_fold` for a splitter-compatible implementation.

**Rule 3 — Build deterministic calendar blocks on the full frame; refit data-estimated
blocks per fold.** Fourier terms and holiday flags are functions of the date. Group means of
the target are not.

**Rule 4 — Slice the error.** Overall WMAPE can hide that one block helps the average while
quietly hurting the slice that matters most.

## 12. Key Takeaways

- **Error concentrates on events** — holiday windows run roughly twice the error of the
  overall average on this panel; event features recover most of that gap
- **Moving events need event features**, not day-of-year Fourier terms
- A **flag** gets *some* of the gain; **countdown / count-up** usually gets more — don't
  assume which until you've measured it on your own data
- **Categoricals matter when history cannot speak** — in true cold start, expanding target
  encoding wins by a distance over every other encoding tested here
- **Target encoding is a fitted feature.** Fitting it before the split can push validation
  error below the noise floor — the diagnostic for a leak
- **Blocks that help alone can hurt together.** Check every addition against the table, not
  against what worked in an earlier article
- Every block in the final matrix has a **leakage rule**; the ledger is the checklist

---

## The Series, Complete

| # | Topic | Notebook |
|---|---|---|
| 1 | Why Time Series Needs Different Features | [`01_tabularizing_time_series.ipynb`](01_tabularizing_time_series.ipynb) |
| 2 | Stationarity & Transformations | [`02_ml_pipeline_leakage.ipynb`](02_ml_pipeline_leakage.ipynb) |
| 3 | Leakage & Time-Aware Validation | [`03_decomposition.ipynb`](03_decomposition.ipynb) |
| 4 | Missing Data Beyond Simple Imputation | [`04_missing_data.ipynb`](04_missing_data.ipynb) |
| 5 | Outlier Detection in Time Series | [`05_outlier_detection.ipynb`](05_outlier_detection.ipynb) |
| 6 | Lag Features & Autocorrelation | [`06_lag_features.ipynb`](06_lag_features.ipynb) |
| 7 | Window (Rolling) Features | [`07_window_features.ipynb`](07_window_features.ipynb) |
| 8 | Trend Features | [`08_trend_features.ipynb`](08_trend_features.ipynb) |
| 9 | Seasonality Features | [`09_seasonality.ipynb`](09_seasonality.ipynb) |
| 10 | Datetime & Categorical Features | `10_datetime_categorical.ipynb` (this notebook) |

Ten articles, one thread: **the model is rarely the bottleneck. The features — and the
discipline about when each one is known — are.**